In [ ]:
# Select a category and restart the kernel before processing another dataset.
CATEGORY = "clothing"  # clothing, electronics, movies, musical
RAW_FILES = {
    "clothing": "reviews_Clothing_Shoes_and_Jewelry.json.gz",
    "electronics": "reviews_Electronics.json.gz",
    "movies": "reviews_Movies_and_TV.json.gz",
    "musical": "reviews_Musical_Instruments.json.gz",
}
if CATEGORY not in RAW_FILES:
    raise ValueError(f"Unsupported category: {CATEGORY}")


In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import time
import torch
import pandas as pd
import gzip
import json

from collections import defaultdict
from torch_geometric.utils.convert import from_networkx
from torch_geometric.utils import to_undirected


%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import torch
import random

def set_seed(seed=42):
    """Set the same seed for reproducibility across NumPy, PyTorch, and Python."""
    np.random.seed(seed)             # NumPy
    torch.manual_seed(seed)          # PyTorch CPU
    torch.cuda.manual_seed(seed)     # PyTorch GPU (if applicable)
    torch.cuda.manual_seed_all(seed) # If using multiple GPUs
    random.seed(seed)                # Python's built-in random module

    # Ensure deterministic behavior in PyTorch
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
SEED = 42
MIN_REVIEW = 5
MIN_SHARED_PRODUCT = 5 if CATEGORY == "movies" else 3

In [ ]:
set_seed(seed=SEED)

In [ ]:

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)
    
def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')



In [ ]:
import os
os.getcwd()

In [ ]:
df = getDF(f"../data/raw/{RAW_FILES[CATEGORY]}")


In [ ]:
df

In [ ]:
len(df['reviewerID'].unique())

In [ ]:
dfc = df.groupby('reviewerID')['asin'].count().reset_index()

dfcf = dfc[dfc['asin'] >= MIN_REVIEW]
print(dfcf)
print(dfcf['asin'].sum())
print(dfcf['asin'].sum()/len(dfcf))

In [ ]:
dff = df[df['reviewerID'].isin(dfcf['reviewerID'])]
dff2 = dff[['reviewerID', 'asin', 'unixReviewTime']]
dff_join = dff2.merge(dff2, on='asin', how='inner')
dff_join['unixReviewTime'] = np.maximum(dff_join['unixReviewTime_x'], dff_join['unixReviewTime_y'])
dff_join

In [ ]:
dff_join_g = dff_join.groupby(['reviewerID_x', 'reviewerID_y']).agg({'asin':'count', 'unixReviewTime': 'min'}).reset_index()

dff_join_f = dff_join_g[(dff_join_g['reviewerID_x'] !=  dff_join_g['reviewerID_y']) & (dff_join_g['asin'] >= MIN_SHARED_PRODUCT)]
dff_join_f

In [ ]:
dff

In [ ]:
## Convert it to personal events

In [ ]:
dffs = dff.sort_values(['reviewerID', 'unixReviewTime']).reset_index(drop=True)
dffs['rating'] = dffs['overall'].astype('int')

dffs['uid'] = pd.factorize(dffs['reviewerID'])[0]
# unique_mapping = {
#     name: f"prod{str(i).zfill(5)}"
#     for i, name in enumerate(dffs['asin'].unique(), start=1)
# }
# dffs['prod_id'] = dffs['asin'].map(unique_mapping)


dffs["event_set"] = 'personal'
# dffs['event'] = dffs['prod_id'] + ':' + dffs['rating'].astype('str')
dffs['event'] = dffs['asin'] + ':' + dffs['rating'].astype('str')

dffs["timestamp"] = dffs['unixReviewTime']
dffs["other_uid"] = None

In [ ]:
dffs['asin'].unique()

In [ ]:
dffs

In [ ]:
personal_columns = ['uid', 'timestamp', 'event_set', 'event', 'reviewText', 'other_uid']
if CATEGORY == "musical":
    personal_columns = ['uid', 'timestamp', 'reviewText', 'event_set', 'event', 'other_uid']
dfp = dffs[personal_columns].sort_values(by=["uid", "timestamp", "event_set"]).reset_index(drop=True)


In [ ]:
dfp

In [ ]:
df_id = dffs[['reviewerID', 'uid']].drop_duplicates()
dff_join_f1 = dff_join_f.merge(df_id, left_on='reviewerID_x', right_on='reviewerID', how='left')
dff_join_f1
dff_join_f2 = dff_join_f1.merge(df_id.rename(columns={'reviewerID': 'reviewerID_y', 'uid': 'other_uid'}), on='reviewerID_y', how='left')
dff_join_f2

In [ ]:
dff_join_f2["event_set"] = 'relational'
dff_join_f2['event'] = 'co-review_product'
dff_join_f2["timestamp"] = dff_join_f2['unixReviewTime']


In [ ]:
dff_join_f2

In [ ]:
dfr = dff_join_f2[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].sort_values(by=["uid", "timestamp", "event_set"]).reset_index(drop=True)
dfr

In [ ]:
df_all = pd.concat([dfp, dfr]).drop_duplicates().sort_values(by=["uid", "timestamp", "event_set", 'other_uid']).reset_index(drop=True)
df_all

In [ ]:
df_stats = df_all.groupby(['uid', 'event_set'])['event'].count().reset_index().sort_values('event')
df_stats

In [ ]:
dfg = df_stats.groupby(['uid'])['event_set'].count().reset_index()
id_have_2set = dfg[dfg['event_set'] == 2]['uid'].values
id_have_2set

In [ ]:
df_all

In [ ]:
df_final = df_all[df_all['uid'].isin(id_have_2set)].reset_index(drop=True)
df_final

In [ ]:
df_stats

In [ ]:
df_stats.groupby(['event_set'])['event'].sum().reset_index()

In [ ]:
df_final.groupby(['uid', 'event_set'])['event'].count().reset_index().sort_values('event').groupby(['event_set'])['event'].sum().reset_index()

In [ ]:
df_all.to_csv(f'../data/processed/amazon-{CATEGORY}_all_events.csv', index=False)


In [ ]:
df_final.to_csv(f'../data/processed/amazon-{CATEGORY}_accounts_with_pr.csv', index=False)


In [ ]:
df_stats.groupby(['event_set'])['uid'].count().reset_index()

In [ ]:
df_final.groupby(['event_set'])['uid'].count().reset_index()

## label

In [ ]:
dffs['helpful_num'] = dffs['helpful'].apply(lambda x: x[0])
dffs['helpful_den'] = dffs['helpful'].apply(lambda x: x[1])
dffs['helpful_prop'] = dffs['helpful_num'] / dffs['helpful_den'] 

In [ ]:
dffs_user = dffs.groupby('uid').agg({'helpful_num': 'sum', 'helpful_den': 'sum'}).reset_index()
dffs_user

In [ ]:
dffs_user['helpful_prop'] = dffs_user['helpful_num'] / dffs_user['helpful_den'] 
dffs_user['fraud_label'] = -1
dffs_user.loc[(dffs_user['helpful_prop'] < 0.3) & (dffs_user['helpful_den'] >= MIN_REVIEW), 'fraud_label'] = 1
dffs_user.loc[(dffs_user['helpful_prop'] > 0.7) & (dffs_user['helpful_den'] >= MIN_REVIEW), 'fraud_label'] = 0


In [ ]:
dffs_user

In [ ]:
df_label_all = dffs_user[dffs_user['fraud_label'] != -1][['uid', 'fraud_label']].reset_index(drop=True)
df_label_all

In [ ]:
df_label_final = df_label_all[df_label_all['uid'].isin(df_final['uid'].unique())]
df_label_final

In [ ]:
df_label_all.to_csv(f'../data/processed/amazon-{CATEGORY}_all_events_label.csv', index=False)
df_label_final.to_csv(f'../data/processed/amazon-{CATEGORY}_accounts_with_pr_label.csv', index=False)



In [ ]:
df_label_all.groupby('fraud_label')['uid'].count().reset_index()

In [ ]:
df_label_final.groupby('fraud_label')['uid'].count().reset_index()